In [71]:
import os
import geopandas as gpd
import pandas as pd

#Get current working directory
project_folder = os.getcwd()

#Build full path to the GeoPackage
farm_file_path = "C:/Users/Vasmai/dataset/farm_boundaries_sampled_20260320.gpkg"

#Load the spatial farm boundary dataset
farm_boundaries = gpd.read_file(farm_file_path)

#get first few records
print(farm_boundaries.head())

#Display column names
print(farm_boundaries.columns)

#Export the full dataset to CSV
csv_file_path = os.path.join("C:/Users/Vasmai/dataset", "farm_boundaries.csv")

farm_boundaries.to_csv(csv_file_path, index=False)

print(f"\nGeoPackage exported to CSV: {csv_file_path}")
# Check dataset structure, data types, and missing values
farm_boundaries.info()

# Dataset not included in GitHub due to privacy
# Can be downloaded from MS Teams → Planting Optimisation Tool → Datasets → GIS

                name treeo2_id  district subdistrict         suku  \
0     Abel Pereira_1              BAUCAU      BAGUIA     Lavateri   
1     Abel Pereira_2              BAUCAU      BAGUIA     Lavateri   
2    Agostinho Alves            VIQUEQUE  UATUCARBAU  Alaua-Craik   
3  Agostinho Freitas            VIQUEQUE  UATUCARBAU  Alaua-Craik   
4     Amelia Menezes              BAUCAU      BAGUIA  Alaua-Craik   

         aldeia     texture   ph  avg_annual_rainfall_2020-2024  \
0  Onortibalari        Clay  6.2                           1959   
1  Onortibalari        Clay  6.2                           1959   
2      Neolidae  Sandy Loam  8.2                           2021   
3      Neolidae  Sandy Loam  5.9                           2554   
4      Neolidae        Clay  7.0                           2021   

   avg_annual_temperature_2020-2024  elevation          area  area_ha  \
0                              24.0        585   3596.997671     0.36   
1                              24.0 

#### Remove Privacy Information

In [72]:
#getrid of privacy info
cols_to_remove = ['name', 'suku', 'aldeia', 'treeo2_id', 'district', 'subdistrict', 'area']
farm_cleaned = farm_boundaries.drop(columns=cols_to_remove, errors='ignore')

# Preview cleaned dataset
print("privacy info removed:")

# Check cleaned dataset info
farm_cleaned.info()

farm_cleaned.head()

privacy info removed:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 7 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   texture                           3153 non-null   object  
 1   ph                                3153 non-null   object  
 2   avg_annual_rainfall_2020-2024     3200 non-null   int32   
 3   avg_annual_temperature_2020-2024  3192 non-null   float64 
 4   elevation                         3200 non-null   int32   
 5   area_ha                           3200 non-null   float64 
 6   geometry                          3200 non-null   geometry
dtypes: float64(2), geometry(1), int32(2), object(2)
memory usage: 150.1+ KB


,texture,ph,avg_annual_rainfall_2020-2024,avg_annual_temperature_2020-2024,elevation,area_ha,geometry
0,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90..."
1,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90..."
2,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90..."
3,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90..."
4,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90..."


#### Rename Columns

In [73]:
# add farm_id and rename columns
farm_cleaned.insert(0, 'farm_id', range(1, len(farm_boundaries) + 1))
farm_cleaned = farm_cleaned.rename(columns={'avg_annual_rainfall_2020-2024': 'rainfall','avg_annual_temperature_2020-2024': 'temperature'})
farm_cleaned.info()
farm_cleaned.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   farm_id      3200 non-null   int64   
 1   texture      3153 non-null   object  
 2   ph           3153 non-null   object  
 3   rainfall     3200 non-null   int32   
 4   temperature  3192 non-null   float64 
 5   elevation    3200 non-null   int32   
 6   area_ha      3200 non-null   float64 
 7   geometry     3200 non-null   geometry
dtypes: float64(2), geometry(1), int32(2), int64(1), object(2)
memory usage: 175.1+ KB


,farm_id,texture,ph,rainfall,temperature,elevation,area_ha,geometry
0,1,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90..."
1,2,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90..."
2,3,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90..."
3,4,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90..."
4,5,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90..."


#### Missing Values

In [74]:
farm_cleaned.isnull().sum()

farm_id         0
texture        47
ph             47
rainfall        0
temperature     8
elevation       0
area_ha         0
geometry        0
dtype: int64

In [75]:
#Convert 'ph' column to float and round to 1 decimal
farm_cleaned['ph'] = farm_cleaned['ph'].astype(float).round(1)

# Fill missing pH values
farm_cleaned['ph'] = farm_cleaned['ph'].fillna(farm_cleaned['ph'].median())

# Fill missing temperature values
farm_cleaned['temperature'] = farm_cleaned['temperature'].fillna(farm_cleaned['temperature'].median())

# Fill missing texture values
farm_cleaned['texture'] = farm_cleaned['texture'].fillna(farm_cleaned['texture'].mode()[0])

# Check missing values again
print(farm_cleaned.isnull().sum())
print(farm_cleaned.head())

farm_id        0
texture        0
ph             0
rainfall       0
temperature    0
elevation      0
area_ha        0
geometry       0
dtype: int64
   farm_id     texture   ph  rainfall  temperature  elevation  area_ha  \
0        1        Clay  6.2      1959         24.0        585     0.36   
1        2        Clay  6.2      1959         24.0        481     0.48   
2        3  Sandy Loam  8.2      2021         25.0        179     1.18   
3        4  Sandy Loam  5.9      2554         24.0        260     0.46   
4        5        Clay  7.0      2021         25.0        129     2.00   

                                            geometry  
0  MULTIPOLYGON Z (((904783.597 9050919.352 0, 90...  
1  MULTIPOLYGON Z (((905235.474 9051056.651 0, 90...  
2  MULTIPOLYGON Z (((903414.205 9042747.721 0, 90...  
3  MULTIPOLYGON Z (((902007.836 9043097.668 0, 90...  
4  MULTIPOLYGON Z (((904016.568 9042595.439 0, 90...  


#### Data Validation after cleaning data

#### Check datatypes

In [76]:
print(farm_cleaned.dtypes)

farm_id           int64
texture          object
ph              float64
rainfall          int32
temperature     float64
elevation         int32
area_ha         float64
geometry       geometry
dtype: object


#### Missing Values

In [77]:
print(farm_cleaned.isnull().sum())

farm_id        0
texture        0
ph             0
rainfall       0
temperature    0
elevation      0
area_ha        0
geometry       0
dtype: int64


#### Valid Range Check

In [78]:
# Describe
print(farm_cleaned.describe())

           farm_id           ph     rainfall  temperature    elevation  \
count  3200.000000  3200.000000  3200.000000  3200.000000  3200.000000   
mean   1600.500000     6.983094  1807.756562    24.194063   517.382187   
std     923.904757     0.879689   347.589473     1.457589   254.967481   
min       1.000000     5.600000   865.000000    19.000000     9.000000   
25%     800.750000     6.200000  1529.000000    23.000000   353.000000   
50%    1600.500000     6.900000  1734.000000    24.000000   489.500000   
75%    2400.250000     7.900000  2034.000000    25.000000   711.000000   
max    3200.000000     8.500000  2624.000000    28.000000  1313.000000   

           area_ha  
count  3200.000000  
mean      1.413378  
std       2.974257  
min       0.000000  
25%       0.390000  
50%       0.750000  
75%       1.410000  
max      85.760000  


#### Replaced "area_ha" with median

In [79]:
median_area = farm_cleaned['area_ha'].median()
farm_cleaned['area_ha'] = farm_cleaned['area_ha'].replace(0, median_area)
print(median_area)

print((farm_cleaned['area_ha'] == 0).sum())

0.75
0


In [80]:
print(farm_cleaned.describe())

           farm_id           ph     rainfall  temperature    elevation  \
count  3200.000000  3200.000000  3200.000000  3200.000000  3200.000000   
mean   1600.500000     6.983094  1807.756562    24.194063   517.382187   
std     923.904757     0.879689   347.589473     1.457589   254.967481   
min       1.000000     5.600000   865.000000    19.000000     9.000000   
25%     800.750000     6.200000  1529.000000    23.000000   353.000000   
50%    1600.500000     6.900000  1734.000000    24.000000   489.500000   
75%    2400.250000     7.900000  2034.000000    25.000000   711.000000   
max    3200.000000     8.500000  2624.000000    28.000000  1313.000000   

           area_ha  
count  3200.000000  
mean      1.414081  
std       2.974011  
min       0.010000  
25%       0.390000  
50%       0.750000  
75%       1.410000  
max      85.760000  


#### Categorical validation

In [81]:
print(farm_cleaned['texture'].unique())

['Clay' 'Sandy Loam' 'Clay Loam' 'Loam' 'Variable' 'Silty Loam'
 'Sandy Clay' 'Organic' 'Silty Clay']


### Spatial Consistency Check

#### Check missing geometries

In [82]:
missing_farm_boundaries = farm_boundaries.geometry.isnull().sum()
missing_farm_cleaned = farm_cleaned.geometry.isnull().sum()
print(missing_farm_boundaries)
print(missing_farm_cleaned)

0
0


#### Check for duplicate farm_id

In [92]:
# Check duplicates in farm_cleaned
print(farm_cleaned['farm_id'].duplicated().sum())

# Check duplicates in farm_boundaries
print(farm_boundaries['farm_id'].duplicated().sum())

0
0


#### Check Geometry validity

In [84]:
print(farm_boundaries.geometry.is_valid.value_counts())

True    3200
Name: count, dtype: int64


#### One-to-One 

In [85]:
num_cleaned_records = len(farm_cleaned)
num_boundary_records = len(farm_boundaries)
one_to_one_issue = num_cleaned_records != num_cleaned_records
print("One-to-one relationship :", one_to_one_issue, "\n")

One-to-one relationship : False 



#### Area Mismatch

In [86]:
farm_boundaries = farm_boundaries.reset_index(drop=True)
farm_boundaries['farm_id'] = range(1, len(farm_boundaries)+1)

farm_boundaries = farm_boundaries.to_crs(epsg=3857)  # meters for accurate area
farm_boundaries['calc_area_ha'] = farm_boundaries.geometry.area / 10000  # hectares

# Merge with master table
check_area = farm_cleaned.merge(farm_boundaries[['farm_id','calc_area_ha']], on='farm_id')
check_area['abs_diff'] = (check_area['calc_area_ha'] - check_area['area_ha']).abs()
check_area['pct_diff'] = (check_area['abs_diff'] / check_area['area_ha']) * 100

area_mismatch = check_area[check_area['pct_diff'] > 5]
print(f"Number of farms with significant area mismatch (>5%): {len(area_mismatch)}")
print(area_mismatch[['farm_id','area_ha','calc_area_ha','abs_diff','pct_diff']].head(10))

Number of farms with significant area mismatch (>5%): 75
     farm_id  area_ha  calc_area_ha  abs_diff   pct_diff
6          7     0.11      0.116325  0.006325   5.750152
216      217     0.24      0.252111  0.012111   5.046115
241      242     0.15      0.159124  0.009124   6.082805
244      245     0.01      0.012899  0.002899  28.993789
355      356     0.21      0.220670  0.010670   5.080890
516      517     0.27      0.283822  0.013822   5.119134
552      553     0.03      0.027198  0.002802   9.341491
560      561     0.14      0.148265  0.008265   5.903897
648      649     0.12      0.126593  0.006593   5.494446
657      658     0.08      0.085566  0.005566   6.958068


#### Overlap

In [87]:
import geopandas as gpd

# Spatial join: find intersecting geometries
joined = gpd.sjoin(farm_boundaries, farm_boundaries, how='inner', predicate='intersects')

# Remove self-matches (same farm_id)
overlaps = joined[joined['farm_id_left'] != joined['farm_id_right']]

print("Number of overlapping boundaries:", len(overlaps))
print(overlaps[['farm_id_left','farm_id_right']].head())

overlap_pairs = overlaps[['farm_id_left','farm_id_right']].copy()

# Keep only unique pairs 
overlap_pairs = overlap_pairs[
    overlap_pairs['farm_id_left'] < overlap_pairs['farm_id_right']
]

print("Unique overlapping pairs:", len(overlap_pairs))
print(overlap_pairs)

Number of overlapping boundaries: 18
      farm_id_left  farm_id_right
2                3            756
450            451            452
451            452            451
755            756              3
1897          1898           2369
Unique overlapping pairs: 9
      farm_id_left  farm_id_right
2                3            756
450            451            452
1897          1898           2369
1899          1900           2580
1919          1920           2580
2149          2150           2551
2158          2159           2567
2174          2175           2567
2422          2423           2435


In [88]:
report_folder = "C:/Users/Vasmai/dataset/reports"
os.makedirs(report_folder, exist_ok=True)

# Save overlaps report
overlaps_unique[['farm_id_left', 'farm_id_right']].to_csv(
    os.path.join(report_folder, "farm_overlaps_report.csv"), index=False
)

# Save area mismatch report
area_mismatch[['farm_id','area_ha','calc_area_ha','abs_diff','pct_diff']].to_csv(
    os.path.join(report_folder, "area_mismatch_report.csv"), index=False
)

print("\nReports saved to:", report_folder)


Reports saved to: C:/Users/Vasmai/dataset/reports


#### Core Engineered features

In [89]:
farm_cleaned['coastal'] = ((farm_cleaned['elevation'] < 100) &
                              (farm_cleaned['rainfall'].between(500, 3500)).astype(bool))
farm_cleaned.head()

,farm_id,texture,ph,rainfall,temperature,elevation,area_ha,geometry,coastal
0,1,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90...",False
1,2,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90...",False
2,3,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90...",False
3,4,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90...",False
4,5,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90...",False


In [90]:
!pip install rasterio rasterstats

#### Finding Slope

In [91]:
import numpy as np
from rasterstats import zonal_stats
import rasterio

print("Farms CRS:", farm_cleaned.crs)

dem_path = "C:/Users/Vasmai/dataset/DEM.tif"

with rasterio.open(dem_path) as dem:
    print("DEM CRS:", dem.crs)
    
    dem_data = dem.read(1)
    dem_transform = dem.transform
    dem_nodata = dem.nodata

# Reproject farms to EPSG:3857 (to match DEM)
farms_reprojected = farm_cleaned.to_crs(epsg=3857)

print("Reprojected farms CRS:", farms_reprojected.crs)

# Convert DEM to float & handle nodata
dem_float = dem_data.astype(float)
dem_float[dem_float == dem_nodata] = np.nan

# Resolution
dem_res_x = dem_transform.a
dem_res_y = abs(dem_transform.e)

# Gradient
dx, dy = np.gradient(dem_float, dem_res_x, dem_res_y)

# Slope in degrees
slope_degrees = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

# Zonal statistics
slope_stats = zonal_stats(
    farms_reprojected.geometry,
    slope_degrees,
    affine=dem_transform,
    stats=["mean"],
    nodata=np.nan,
    all_touched=True
)

# Assign slope
farms_reprojected["slope"] = [s["mean"] for s in slope_stats]
farms_reprojected["slope"].describe()

# Reproject and overwrite farm_cleaned
farm_cleaned = farm_cleaned.to_crs(epsg=3857)

# Then do zonal stats
slope_stats = zonal_stats(
    farm_cleaned.geometry,
    slope_degrees,
    affine=dem_transform,
    stats=["mean"],
    nodata=np.nan,
    all_touched=True
)

# Save slope directly
farm_cleaned["slope"] = [s["mean"] for s in slope_stats]
print(farm_cleaned[["farm_id","slope"]].head(10))
farm_cleaned.head()

Farms CRS: EPSG:32751
DEM CRS: LOCAL_CS["WGS 84 / Pseudo-Mercator",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Reprojected farms CRS: EPSG:3857
   farm_id      slope
0        1  13.114323
1        2  13.937272
2        3  11.151526
3        4  15.552424
4        5   7.615144
5        6   9.167869
6        7  30.078060
7        8  17.737309
8        9   7.997971
9       10   7.479812


,farm_id,texture,ph,rainfall,temperature,elevation,area_ha,geometry,coastal,slope
0,1,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((14101512.97 -957413.083 0, 1...",False,13.114323
1,2,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((14101967.821 -957269.166 0, ...",False,13.937272
2,3,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((14100209.583 -965731.093 0, ...",False,11.151526
3,4,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((14098786.13 -965389.142 0, 1...",False,15.552424
4,5,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((14100819.288 -965879.974 0, ...",False,7.615144


#### Adding Latitude and Longitude

In [93]:
import geopandas as gpd

# Convert to GeoDataFrame if it's a plain DataFrame
if not isinstance(farm_cleaned, gpd.GeoDataFrame):
    farm_cleaned = gpd.GeoDataFrame(
        farm_cleaned,
        geometry='geometry',  # <-- replace with your geometry column name
        crs='EPSG:32751'      # <-- replace with your current CRS
    )

# Compute centroids in meters
centroids_proj = farm_cleaned.geometry.centroid

# If you want lon/lat, reproject back to EPSG:4326
farm_cleaned['longitude'] = centroids_proj.to_crs(epsg=4326).x
farm_cleaned['latitude'] = centroids_proj.to_crs(epsg=4326).y

# Show first 5 farms
farm_cleaned[['farm_id', 'longitude', 'latitude']].head()
farm_cleaned.head()

,farm_id,texture,ph,rainfall,temperature,elevation,area_ha,geometry,coastal,slope,longitude,latitude
0,1,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((14101512.97 -957413.083 0, 1...",False,13.114323,126.675704,-8.568798
1,2,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((14101967.821 -957269.166 0, ...",False,13.937272,126.680149,-8.567578
2,3,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((14100209.583 -965731.093 0, ...",False,11.151526,126.664651,-8.642259
3,4,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((14098786.13 -965389.142 0, 1...",False,15.552424,126.651458,-8.639690
4,5,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((14100819.288 -965879.974 0, ...",False,7.615144,126.670084,-8.643001


#### Riparian

In [94]:
import geopandas as gpd

# Load waterways
waterways_path = "C:/Users/Vasmai/dataset/hotosm_tls_waterways_lines_gpkg.gpkg"
waterways = gpd.read_file(waterways_path)
waterways = waterways.to_crs(farm_cleaned.crs)

# Create 15m buffer
waterways_buffer = waterways.buffer(15)
waterways_buffer_gdf = gpd.GeoDataFrame(geometry=waterways_buffer)

# Spatial join: which farms intersect any buffer
joined = gpd.sjoin(farm_cleaned, waterways_buffer_gdf, how="left", predicate="intersects")

# Initialize column as False
farm_cleaned['riparian'] = False

# Mark True only for farms that intersect
farm_cleaned.loc[joined.index.unique(), 'riparian'] = True

# Check
print(farm_cleaned[['farm_id', 'riparian']].head(10))

   farm_id  riparian
0        1      True
1        2      True
2        3      True
3        4      True
4        5      True
5        6      True
6        7      True
7        8      True
8        9      True
9       10      True


In [95]:
# CSV file path
csv_file_path = os.path.join("C:/Users/Vasmai/dataset", "farm_boundaries_cleaned.csv")

# Save cleaned DataFrame directly to CSV
farm_cleaned.to_csv(csv_file_path, index=False) 

print(f"\nCleaned dataset exported to CSV: {csv_file_path}")


Cleaned dataset exported to CSV: C:/Users/Vasmai/dataset\farm_boundaries_cleaned.csv


In [97]:
print(farm_cleaned.head())

   farm_id     texture   ph  rainfall  temperature  elevation  area_ha  \
0        1        Clay  6.2      1959         24.0        585     0.36   
1        2        Clay  6.2      1959         24.0        481     0.48   
2        3  Sandy Loam  8.2      2021         25.0        179     1.18   
3        4  Sandy Loam  5.9      2554         24.0        260     0.46   
4        5        Clay  7.0      2021         25.0        129     2.00   

                                            geometry  coastal      slope  \
0  MULTIPOLYGON Z (((14101512.97 -957413.083 0, 1...    False  13.114323   
1  MULTIPOLYGON Z (((14101967.821 -957269.166 0, ...    False  13.937272   
2  MULTIPOLYGON Z (((14100209.583 -965731.093 0, ...    False  11.151526   
3  MULTIPOLYGON Z (((14098786.13 -965389.142 0, 1...    False  15.552424   
4  MULTIPOLYGON Z (((14100819.288 -965879.974 0, ...    False   7.615144   

    longitude  latitude  riparian  
0  126.675704 -8.568798      True  
1  126.680149 -8.567578   